# Decision Tree — Scheme 2 (Static Test) — LUMED

> Run on **Google Colab**. Mount your Google Drive and adjust `folder_path` before executing.


In [ ]:
import os, numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Define training size (change per experiment step) ──
train_size = 0.95  # fraction of training pool used at this step

folder_path  = '/content/drive/My Drive/EEG Datasets/LUMED/LUMED CSV/'
testset_file = 'lumed_testset.csv'

data = pd.DataFrame()
csv_files = [f"wavelet_denoised_s{i:02}.csv" for i in range(1, 14)]
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    temp_data = pd.read_csv(file_path)
    temp_data = temp_data.sample(frac=0.2, random_state=42)
    data = pd.concat([data, temp_data], ignore_index=True)

# ── Load the FIXED static test set (never subsampled) ──
test_data = pd.read_csv(os.path.join(folder_path, testset_file))

# ── Subsample training pool ──
data_train = data.sample(frac=train_size, random_state=42).reset_index(drop=True)

X_train_raw = data_train.drop(columns=['label']).values
y_train_raw = data_train['label'].apply(lambda x: eval(x)[0]).values
X_test_raw  = test_data.drop(columns=['label']).values
y_test_raw  = test_data['label'].apply(lambda x: eval(x)[0]).values

# ── Fit scaler on TRAINING data ONLY ──
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)
y_train, y_test = y_train_raw, y_test_raw

# ────────────────────────────────────────────────────────────
# Decision Tree — Scheme 2 (Static Test) — LUMED
# ────────────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion='gini', splitter='best', max_depth=10, random_state=42)

# ── 5-Fold Cross-Validation on training data ──
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy with {int(train_size * 100)}% training data: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} (SD)")

model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
print(f"Held-out Test Accuracy with {int(train_size * 100)}% training data: {accuracy:.4f}")

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, normalize='true')
plt.figure(figsize=(5, 4))
sns.heatmap(cm * 100, annot=True, fmt=".2f", cmap="Blues")
plt.title(f"Confusion Matrix ({int(train_size * 100)}% Training Data)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred))